In [1]:
from pyspark.sql import SparkSession
#import google.generativeai as genai
from com.example.ai.LLMManager import LLMManager

model = LLMManager.get_model()
# Configure Gemini API
#genai.configure(api_key="your_api_key")

# Initialize Spark session
# spark = SparkSession.builder.appName("NLtoSparkSQL").getOrCreate()

Step 2: Generating Sample Data

In [2]:
import builtins
import random
from datetime import datetime, timedelta

data = []
num_records = 1000  # Number of records

for _ in range(num_records):
    order_id = random.randint(1000, 9999)
    customer_id = random.randint(100, 500)
    product_id = random.randint(1, 20)
    product_names = ["Laptop", "Smartphone", "T-Shirt", "Book", "Headphones"]
    product_name = random.choice(product_names)
    categories = ["Electronics", "Clothing", "Books"]
    category = random.choice(categories)
    start_date = datetime(2022, 1, 1)
    order_date = start_date + timedelta(days=random.randint(0, 730))
    quantity = random.randint(1, 5)
    price = builtins.round(random.uniform(10, 500), 2)
    regions = ["North", "South", "East", "West"]
    region = random.choice(regions)
    data.append((order_id, customer_id, product_id, product_name, category, order_date, quantity, price, region))

Step 3: Creating a DataFrame

In [3]:
# Initialize Spark session
# spark = SparkSession.builder.appName("NLtoSparkSQL").getOrCreate()
df = spark.createDataFrame(data, ["order_id", "customer_id", "product_id", "product_name", "category", "order_date", "quantity", "price", "region"])
df.createOrReplaceTempView("sales_data")
df.show(5)

+--------+-----------+----------+------------+-----------+-------------------+--------+------+------+
|order_id|customer_id|product_id|product_name|   category|         order_date|quantity| price|region|
+--------+-----------+----------+------------+-----------+-------------------+--------+------+------+
|    6313|        167|        11|  Smartphone|Electronics|2023-03-09 00:00:00|       3|231.21| South|
|    7710|        369|        18|  Smartphone|      Books|2023-09-27 00:00:00|       2|128.75|  West|
|    7138|        128|        19|  Headphones|   Clothing|2023-05-31 00:00:00|       4|182.31|  West|
|    1151|        318|        17|      Laptop|   Clothing|2022-09-15 00:00:00|       2|118.76| North|
|    3619|        396|        15|     T-Shirt|   Clothing|2023-01-13 00:00:00|       5|199.13| South|
+--------+-----------+----------+------------+-----------+-------------------+--------+------+------+
only showing top 5 rows



Step 4: Implementing the Natural Language to Spark SQL Conversion

In [4]:
def generate_spark_sql(natural_language_query, table_name):
    """Generates Spark SQL from natural language using Gemini."""
    try:
        #model = genai.GenerativeModel("gemini-1.5-pro-latest")
        prompt = f"Given a table named {table_name}, convert the following natural language query to Spark SQL. Only return the SQL code, do not add any markdown code blocks: '{natural_language_query}'"
        response = model.invoke(prompt)
        sql_query = response.text.strip().replace("```sql", "").replace("```", "").strip()
        return sql_query
    except Exception as e:
        print(f"Error generating SQL: {e}")
        return None

Step 5: Executing the Generated SQL Query


In [5]:
def execute_spark_sql(spark, sql_query, table_name):
    """Executes Spark SQL and returns the results."""
    try:
        modified_sql = sql_query.replace("sales_table", table_name)  # Dynamically replace table name
        result_df = spark.sql(modified_sql)
        return result_df.show()
    except Exception as e:
        print(f"Error executing SQL: {e}")
        return None

Step 6: Running the Pipeline

In [10]:
natural_query = "What are the top 5 categories?"
table_name = "sales_data"

sql_query = generate_spark_sql(natural_query, table_name)
print(f"AI Generated Spark SQL Query : {sql_query}")
if sql_query:
    execute_spark_sql(spark, sql_query, table_name)

AI Generated Spark SQL Query : SELECT category, COUNT(category) FROM sales_data GROUP BY category ORDER BY COUNT(category) DESC LIMIT 5
+-----------+---------------+
|   category|count(category)|
+-----------+---------------+
|   Clothing|            341|
|Electronics|            330|
|      Books|            329|
+-----------+---------------+



Step 7: Stop Spark Session

In [ ]:
spark.stop()